# SnowPro Core Certification Breakdown

| Domain                                                 | Weight |
| ------------------------------------------------------ | ------ |
| 1. Snowflake AI Data Cloud Features & Architecture     | 31%    |
| 2. Account Management & Data Governance                | 20%    |
| 3. Data Loading, Unloading & Connectivity              | 18%    |
| 4. Performance Optimization, Querying & Transformation | 21%    |
| 5. Data Collaboration                                  | 10%    |

# SnowPro Core Certification - Complete Study Notes

---

## Domain 1: Snowflake AI Data Cloud Features & Architecture (31%)

---

### 1.1 Snowflake Architecture Overview

**Snowflake architecture**

Snowflake’s architecture is a hybrid of traditional shared-disk and shared-nothing database architectures. Similar to shared-disk architectures, Snowflake uses a central data repository for persisted data that is accessible from all compute nodes in the platform. But similar to shared-nothing architectures, Snowflake processes queries using massively parallel processing (MPP) compute clusters, where each node in the cluster stores a portion of the entire data set locally.

> shared-disk architecture: a central data repository for persisted data that is accessible from all compute nodes in the Snowflake.

> shared-nothing architecture: each node in the cluster stores a portion of the entire data set locally. data is not shared among nodes.

**Three-Layer Architecture:**

| Layer | Purpose | Key Points |
|-------|---------|------------|
| **Cloud Services** | Brain of Snowflake — authentication, metadata, query optimization, access control | Always running, shared across all users |
| **Query Processing (Compute)** | Virtual Warehouses execute queries | Independent scaling, no shared resources between warehouses |
| **Database Storage** | Centralized, columnar, compressed storage | Data stored in micro-partitions, immutable |

**Key Architectural Principles:**
- **Multi-cluster shared data architecture** — separates storage from compute
- Storage and compute scale independently
- Multiple warehouses can access same data simultaneously without contention
- Metadata stored in Cloud Services layer (not in customer storage)

### 1.2 Cloud Platforms & Editions

**Supported Cloud Platforms:** AWS, Azure, Google Cloud Platform

**Editions (lowest to highest):**

| Edition | Key Features Added |
|---------|-------------------|
| **Standard** | Basic features, Time Travel (1 day), always-on encryption |
| **Enterprise** | Multi-cluster warehouses, 90-day Time Travel, materialized views, column-level security, data masking, row access policies, search optimization |
| **Business Critical** | HIPAA/PCI compliance, Tri-Secret Secure, failover/failback, private connectivity (AWS PrivateLink, Azure Private Link, GCP Private Service Connect) |
| **Virtual Private Snowflake (VPS)** | Dedicated metadata store, completely isolated environment |

**IMPORTANT for exam:** Know which features are available in which edition!

### 1.3 Micro-Partitions & Data Clustering

**Micro-Partitions:**
- Data is automatically divided into micro-partitions (50-500 MB compressed)
- Stored in columnar format
- Immutable — any DML creates new micro-partitions
- Metadata stored: range of values, number of distinct values, NULL count per column

**Clustering:**
- Snowflake automatically clusters data as it's ingested (natural clustering)
- **Clustering Key** — user-defined column(s) to reorganize data for better pruning
- Best for very large tables (multi-TB) with known filter patterns
- Automatic Clustering (background service) maintains clustering over time
- **Clustering Depth** — measures how well a table is clustered (lower = better)
- Use `SYSTEM$CLUSTERING_INFORMATION('table_name')` to check clustering quality

**Partition Pruning:**
- Snowflake uses micro-partition metadata to skip irrelevant partitions
- This is the PRIMARY performance optimization mechanism
- Works best with range/equality filters on clustered columns

### 1.4 Data Types

| Category | Types |
|----------|-------|
| Numeric | NUMBER, DECIMAL, INT, BIGINT, FLOAT, DOUBLE |
| String | VARCHAR, CHAR, STRING, TEXT |
| Binary | BINARY, VARBINARY |
| Date/Time | DATE, TIME, TIMESTAMP, TIMESTAMP_LTZ, TIMESTAMP_NTZ, TIMESTAMP_TZ |
| Semi-structured | VARIANT, OBJECT, ARRAY |
| Boolean | BOOLEAN |
| Geospatial | GEOGRAPHY, GEOMETRY |

**VARIANT** — stores semi-structured data (JSON, Avro, Parquet, ORC, XML) up to 16 MB per row.

### 1.5 Encryption

In Snowflake, data is encrypted both **at rest** and **in transit** by default. Encryption is automatic and **cannot be disabled** by customers.

---

#### 1. Encryption at Rest

**Definition:** Data is encrypted while stored in Snowflake, including:
- Tables
- Internal stages
- Micro-partitions
- Backups
- Metadata

**How Snowflake Does It:**
- Uses **AES-256** encryption
- Implements a hierarchical key management model (File Key → Table Master Key → Account Master Key → Root Key)
- Encryption keys are automatically managed and rotated by Snowflake
- Data is encrypted **before** being written to storage

> **In short:** All stored data is automatically encrypted using AES-256. Even if the underlying cloud storage is compromised, the data cannot be read without the encryption keys.

---

#### 2. Encryption in Transit

**Definition:** Data is encrypted while moving between:
- Client and Snowflake
- Snowflake internal services
- Snowflake and cloud storage

**How Snowflake Does It:**
- Uses **TLS (Transport Layer Security) 1.2** or higher
- Protects data from interception during transmission
- All connections (JDBC, ODBC, Snowsight, etc.) are encrypted automatically

> **In short:** All network communication is encrypted using TLS, preventing unauthorized interception of data during transmission.

---

#### Summary

| Feature | Encryption at Rest | Encryption in Transit |
|---------|---|---|
| Protects stored data | Yes | No |
| Protects data in motion | No | Yes |
| Algorithm / Protocol | AES-256 | TLS 1.2+ |
| Enabled by default | Yes | Yes |
| Can be disabled | No | No |

---

**Q. What does "data encrypted at rest and in transit" mean in Snowflake?**

> **Ans:** Snowflake secures data in two ways:
> - **At Rest:** All stored data is automatically encrypted using AES-256.
> - **In Transit:** All data exchanged between clients, Snowflake services, and storage is protected using TLS 1.2+.
>
> This provides end-to-end security without requiring any manual configuration.

### 1.6 Key Rotation & Re-Keying

**Hierarchical Key Model:**
```
Root Key (held by Snowflake or customer)
  └── Account Master Key
        └── Table Master Keys
              └── File Keys (one per micro-partition file)
```

**Key Rotation:**
- Snowflake automatically rotates keys **every 30 days**
- When a key is rotated:
  - A new version of the key is created
  - The NEW key encrypts new data going forward
  - OLD data is NOT immediately re-encrypted (still accessible via old key version)
- This is **automatic** in all editions

**Periodic Re-Keying (Re-encryption):**
- Available in **Enterprise edition and above**
- After key rotation, old data is **re-encrypted** with the new key
- Must be enabled **manually**
- This happens automatically in the background
- Ensures retired keys are no longer needed to decrypt ANY data
- Difference from rotation: Rotation = new key for new data. Re-keying = old data re-encrypted with new key.

**Tri-Secret Secure (Business Critical+):**
- Customer provides their own key (via cloud KMS: AWS KMS, Azure Key Vault, GCP Cloud KMS)
- Composite master key = Customer key + Snowflake key
- Both keys needed to access data
- Customer can revoke their key to make data inaccessible
- This gives customer full control over their data

| Concept | What Happens | Edition | Frequency |
|---------|-------------|---------|----------|
| Encryption | AES-256 on all data | All | Always on |
| Key Rotation | New key version created, new data uses new key | All | Every 30 days (auto) |
| Re-Keying | Old data re-encrypted with new key | Enterprise+ | Automatic background |
| Tri-Secret Secure | Customer-managed key + Snowflake key | Business Critical+ | Customer-controlled |

### 1.7 Caching Layers

| Cache Type | Location | Scope | Persistence | Notes |
|-----------|----------|-------|-------------|-------|
| **Metadata Cache** | Cloud Services | Global | Always available | Stores row count, min/max, etc. |
| **Result Cache** | Cloud Services | Per-user (role) | 24 hours | Same query → instant result, no warehouse needed |
| **Local Disk Cache (Warehouse Cache)** | Compute Layer | Per warehouse | Until warehouse suspends | Raw data cached on SSD of warehouse nodes |

**Result Cache rules:**
- Exact same query text
- Same role
- Underlying data unchanged
- Valid for 24 hours (resets on access, max 31 days)
- No warehouse credits consumed
- Available across virtual warehouses
- Can be disabled per session: `ALTER SESSION SET USE_CACHED_RESULT = FALSE`
- Queries containing non-deterministic functions (e.g., CURRENT_TIME(), CURRENT_TIMESTAMP(), RANDOM()) cannot use Snowflake's result cache because the output may change on every execution.

### 1.8 Continuous Data Protection

**Time Travel:**
- Access historical data (before DML/DDL changes)
- Standard: 0-1 day
- Enterprise+: 0-90 days (configurable via `DATA_RETENTION_TIME_IN_DAYS`)
- Transient/Temporary tables: max 1 day (any edition)
- Use: `SELECT * FROM table AT(OFFSET => -60*5)` or `BEFORE(STATEMENT => 'query_id')`
- Can UNDROP tables, schemas, databases within retention

**Fail-Safe:**
- 7-day period AFTER Time Travel expires
- Only Snowflake Support can recover data (not user-accessible)
- Non-configurable
- Transient and Temporary tables have NO Fail-safe
- Additional storage cost

| Table Type | Time Travel (max) | Fail-Safe |
|-----------|-------------------|----------|
| Permanent | 90 days (Enterprise+) | 7 days |
| Transient | 1 day | None |
| Temporary | 1 day (session only) | None |

### 1.9 Snowflake Cortex AI Features

---

#### AI SQL Functions (Gen2: `AI_*` vs Gen1: `SNOWFLAKE.CORTEX.*`)

Snowflake has two generations of AI function syntax:

| Generation | Syntax | Schema Prefix Required? | Status |
|-----------|--------|------------------------|--------|
| **Gen2 (current)** | `AI_COMPLETE()`, `AI_SENTIMENT()`, etc. | NO — globally available | Preferred |
| **Gen1 (legacy)** | `SNOWFLAKE.CORTEX.COMPLETE()`, `SNOWFLAKE.CORTEX.SENTIMENT()`, etc. | YES | Still supported |

> **Exam rule of thumb:** `AI_*` = Gen2 (current), `SNOWFLAKE.CORTEX.*` = Gen1 (legacy). Both work, but Gen2 is the modern interface.

---

#### Gen2 Cortex AI Functions (use these)

| Function | Purpose |
|----------|--------|
| `AI_COMPLETE()` | Text/image generation using LLMs (supports model selection) |
| `AI_CLASSIFY()` | Classify text or images into user-defined categories |
| `AI_FILTER()` | Returns TRUE/FALSE for text/image — use in WHERE clauses |
| `AI_EXTRACT()` | Extract structured info from text, images, or documents |
| `AI_SENTIMENT()` | Sentiment score from text |
| `AI_TRANSLATE()` | Translate text between languages |
| `AI_PARSE_DOCUMENT()` | OCR / layout extraction from staged documents |
| `AI_REDACT()` | Redact PII from text |
| `AI_TRANSCRIBE()` | Transcribe audio/video files from stage |

**Helper functions:**
- `TO_FILE()` — creates a reference to a staged file for use with AI functions
- `PROMPT()` — builds prompt objects for AI_COMPLETE

---

#### Gen1 Legacy Functions (SNOWFLAKE.CORTEX schema)

| Gen1 (Legacy) | Gen2 (Current) |
|---------------|----------------|
| `SNOWFLAKE.CORTEX.COMPLETE()` | `AI_COMPLETE()` |
| `SNOWFLAKE.CORTEX.SENTIMENT()` | `AI_SENTIMENT()` |
| `SNOWFLAKE.CORTEX.SUMMARIZE()` | `AI_SUMMARIZE_AGG()` |
| `SNOWFLAKE.CORTEX.TRANSLATE()` | `AI_TRANSLATE()` |
| `SNOWFLAKE.CORTEX.EXTRACT_ANSWER()` | `AI_EXTRACT()` |

---

#### How AI Functions Compute (Warehouse + Serverless)

```
┌─────────────────────────────────────────────────────────────────┐
│  YOUR QUERY: SELECT AI_SENTIMENT(review) FROM reviews;          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  [Virtual Warehouse]          [Snowflake AI Infrastructure]     │
│   • Runs the SQL query         • Runs LLM inference (serverless)│
│   • Scans table rows           • Token-based billing            │
│   • Assembles final result     • Warehouse size doesn't help    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

**Two billing components:**

| Component | What It Does | Billed As |
|-----------|-------------|----------|
| Virtual Warehouse | Runs the surrounding SQL (scan, filter, join, output) | Warehouse credits (per-second) |
| AI Inference | Sends prompt to LLM, receives response | Serverless credits (per-token) |

**Key insight:** A warehouse is required to run the query, but the AI inference (the process of sending your prompt to an AI model (LLM) and getting a response back) itself is serverless. Increasing warehouse size beyond Medium does **NOT** speed up the AI function — the LLM inference latency is independent of warehouse size.

**Recommendation:** Use a Small or Medium warehouse for AI function queries. Larger warehouses only help if the surrounding SQL (scanning millions of rows, joins) is the bottleneck — not the AI call itself.

---

#### Usage Example

```sql
SELECT AI_SENTIMENT('This product is amazing!') AS sentiment;

SELECT AI_COMPLETE('llama3.1-70b', 'Explain gravity in one sentence') AS response;

SELECT review_text, AI_SENTIMENT(review_text) AS score FROM customer_reviews;
```

---

#### Privileges Required

Two types of privileges are needed to use AI functions:

| # | Privilege | Level | Granted To | By Default? |
|---|-----------|-------|-----------|-------------|
| 1 | `USE AI FUNCTIONS` | Account-level privilege | PUBLIC | Yes |
| 2 | `CORTEX_USER` or `AI_FUNCTIONS_USER` | Database role (in SNOWFLAKE db) | PUBLIC | Yes |

**Bottom line:** All users can access AI functions out of the box — no setup needed. Access is only restricted if an admin explicitly revokes either privilege.

```sql
-- To restrict access (revoke from PUBLIC, grant to specific roles):
REVOKE USE AI FUNCTIONS ON ACCOUNT FROM ROLE PUBLIC;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE DATA_SCIENCE;
```

---

#### Cortex Search

- Hybrid search (semantic + keyword/BM25) over **unstructured** text data
- Create a Cortex Search Service on a table column
- No need to manage embeddings manually
- Returns relevant results using vector similarity + BM25

#### Cortex Analyst

- Natural language → SQL over **structured** data
- Uses a **Semantic View** (YAML definition of your data model)
- Users ask questions in plain English → generates and runs SQL

#### Cortex Agent

- Orchestration layer that splits complex questions into subtasks
- Routes unstructured parts → Cortex Search, structured parts → Cortex Analyst
- Can call custom tools
- Combines results into a unified answer

#### Cortex Guard

- Safety layer (powered by Meta's Llama Guard) that filters unsafe model responses
- Categories: violent content, hate speech, sexual content, self-harm

#### Cortex Fine-Tuning

- Fine-tune LLMs on your data within Snowflake
- Data never leaves Snowflake
- `SNOWFLAKE.CORTEX.FINETUNE()` function

#### Document AI

- `AI_PARSE_DOCUMENT()` — OCR and layout extraction from PDFs/documents
- `AI_EXTRACT()` — extract structured fields from documents
- Supports multimodal: text, images, PDFs, audio, video

#### Snowpark ML

- ML model training and inference within Snowflake
- **Model Registry** — store, version, deploy models
- **Feature Store** — manage and serve ML features

---

#### Key Exam Points

| Point | Detail |
|-------|--------|
| Data residency | All AI functions run inside Snowflake (data doesn't leave) |
| Infrastructure | Fully managed — no infra to set up |
| Billing | Dual: warehouse credits (SQL) + serverless credits (per-token for inference) |
| Warehouse sizing | Small/Medium recommended — larger doesn't speed up AI inference |
| Regional availability | Some models/functions limited by region; cross-region inference available |
| Privileges | Both granted to PUBLIC by default — all users have access unless revoked |
| Gen2 vs Gen1 | `AI_*` is Gen2 (no prefix), `SNOWFLAKE.CORTEX.*` is Gen1 (legacy) |

---

## Domain 2: Account Management & Data Governance (20%)

---

### 2.1 Authentication Methods

| Method | Description | Key Points |
|--------|-------------|------------|
| **Username/Password** | Default method | Basic authentication |
| **MFA (Multi-Factor Authentication)** | Duo Security push | Powered by Duo; enabled per user; can be enforced via authentication policy |
| **SSO (Single Sign-On)** | SAML 2.0 federation | Snowflake = Service Provider; IdP = Okta, Azure AD, etc. |
| **Key Pair Authentication** | RSA 2048-bit key pair | Used for service accounts, connectors, drivers |
| **OAuth** | External OAuth or Snowflake OAuth | Token-based access for apps |
| **Federated Authentication** | SSO + SCIM | SCIM = auto user/group provisioning from IdP |

**MFA Details:**
- Uses Duo Security (built-in, no extra cost)
- Enrolled per-user: `ALTER USER SET MINS_TO_BYPASS_MFA = ...` is NOT a thing — MFA is enrolled by user
- Users self-enroll via Snowsight OR admin enforces via authentication policy
- Token caching available for programmatic clients (`ALLOW_CLIENT_MFA_CACHING`)
- **Authentication Policy** can REQUIRE MFA for specific users/account
- The SECURITYADMIN role has the privilege to manage and disable MFA on a user basis [Helps in recovering accounts]
- There is no way to prevent a user from enrolling in MFA
- If Authentication Policy enforces to use MFA, enrolling MFA is mandatory.
- If Authentication Policy does not enforce to use MFA, enrolling MFA is optional.

**SSO Details:**
- SAML 2.0
- Configure with `CREATE SECURITY INTEGRATION TYPE = SAML2`
- SSO can be set at account level or user level
- Supports IdP-initiated and SP-initiated flows

### 2.2 Network Policies

- IP allowlist/blocklist for account or user-level access
- `CREATE NETWORK POLICY` with `ALLOWED_IP_LIST` and `BLOCKED_IP_LIST`
- Can be applied to: entire account, specific users, or security integrations
- CIDR notation supported
- Network policies support only IPv4 addresses
- At least one IP must be allowed if policy is active

### 2.3 RBAC (Role-Based Access Control)

**System-Defined Roles (hierarchy, top to bottom):**

```
ORGADMIN          → Manages the organization (multi-account)
    ↑
ACCOUNTADMIN      → Top-level account role (SYSADMIN + SECURITYADMIN)
    ↑
├── SYSADMIN      → Manages all databases, warehouses, objects
│       ↑
│   Custom Roles  → Should be granted to SYSADMIN
│
└── SECURITYADMIN → Manages grants, users, roles
        ↑
    USERADMIN     → Creates users and roles only
        ↑
      PUBLIC      → Auto-granted to every user
```

**Key Rules:**
- All custom roles should be granted to SYSADMIN (best practice)
- ACCOUNTADMIN = SYSADMIN + SECURITYADMIN combined
- ACCOUNTADMIN should have MFA, should NOT be default role
- A user's DEFAULT_ROLE is used on login (if not specified)

**Privileges flow UP the hierarchy** — parent roles inherit all child role privileges.

### 2.4 ACCOUNT_USAGE vs INFORMATION_SCHEMA

| Feature | ACCOUNT_USAGE (Shared DB) | INFORMATION_SCHEMA (Per DB) |
|---------|--------------------------|----------------------------|
| **Location** | `SNOWFLAKE` shared database → `ACCOUNT_USAGE` schema | Each database → `INFORMATION_SCHEMA` schema |
| **Scope** | Entire account (all databases) | Single database only |
| **Latency** | 45 min to 3 hours delay | Real-time (no latency) |
| **Data Retention** | 1 year (365 days) for most views | Varies (7 days to 6 months depending on view) |
| **Dropped Objects** | YES — shows dropped objects | NO — only current/active objects |
| **Access** | ACCOUNTADMIN by default (can grant IMPORTED PRIVILEGES) | Any role with access to the database |
| **Object** | Views | Views (table functions for some) |

**Who can access ACCOUNT_USAGE?**
- ACCOUNTADMIN has access by default
- Other roles need: `GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE <role_name>`
- This grants access to ALL schemas in the SNOWFLAKE database (ACCOUNT_USAGE, READER_ACCOUNT_USAGE, etc.)

**Who can access INFORMATION_SCHEMA?**
- Any role that has privileges on the parent database
- Shows only objects the current role has access to
- No special grant needed beyond database access

**Exam Tip:** If a question asks about historical/dropped data or account-wide → ACCOUNT_USAGE. If real-time/current database → INFORMATION_SCHEMA.

### 2.5 TABLE_STORAGE_METRICS

`TABLE_STORAGE_METRICS` is available in **both** `SNOWFLAKE.ACCOUNT_USAGE` and `DB.INFORMATION_SCHEMA`.

Both versions show storage breakdown per table: `ACTIVE_BYTES`, `TIME_TRAVEL_BYTES`, and `FAILSAFE_BYTES`.

**ACCOUNT_USAGE version** covers all tables across the entire account, includes dropped/deleted tables, has up to 3 hours latency, retains data for 365 days, and requires ACCOUNTADMIN or IMPORTED PRIVILEGES.

**INFORMATION_SCHEMA version** covers only the current database, shows only active (non-dropped) tables, is real-time with no latency, and is accessible by any role that has access to the database.

**Exam Tip:** Dropped tables or account-wide storage → `ACCOUNT_USAGE.TABLE_STORAGE_METRICS`. Current tables in a single database (real-time) → `INFORMATION_SCHEMA.TABLE_STORAGE_METRICS`.

### 2.7 Cloning (Zero-Copy Clone)

**How it works:**
- `CREATE TABLE new_table CLONE source_table`
- Creates a metadata-only copy — no physical data is copied initially
- Both original and clone point to same micro-partitions
- When either is modified, new micro-partitions are created (copy-on-write)
- Storage cost only for changed/new data

**What can be cloned:**

| Object | Cloneable? | Notes |
|--------|-----------|-------|
| Database | YES | Clones all schemas and objects within |
| Schema | YES | Clones all objects within |
| Table (Permanent) | YES | |
| Table (Transient) | YES | |
| Table (Temporary) | YES | Clone becomes Temporary |
| Stream | YES | But becomes empty (no unconsumed records) |
| Stage (Named Internal) | NO | Cannot clone internal stages |
| Stage (External) | YES | Only metadata cloned |
| Pipe | NO | Not cloneable |
| Task | YES | Created in suspended state |
| Sequence | YES | Clone starts at current sequence value |
| File Format | YES | |
| UDF/Procedure | YES | |
| View | YES | |

**Does cloning clone ACCESS PRIVILEGES?**

| Scenario | Privileges Cloned? |
|----------|-------------------|
| `CREATE TABLE ... CLONE` (table only) | **NO** — no privileges are copied |
| `CREATE SCHEMA ... CLONE` | **YES** — all object-level grants within the schema ARE copied |
| `CREATE DATABASE ... CLONE` | **YES** — all schema and object-level grants ARE copied |

**Key Rule:** 
- Cloning a **single object** (table, view) → privileges are NOT copied
- Cloning a **container** (database, schema) → privileges on contained objects ARE copied
- The OWNERSHIP privilege is always granted to the role performing the clone

**Required Privileges for Cloning:**
- Tables: SELECT on source table + CREATE TABLE on target schema
- Schema: CREATE SCHEMA on target database + object privileges
- Database: CREATE DATABASE on account

**Time Travel + Cloning:**
- Can clone from a point in time: `CLONE ... AT(TIMESTAMP => ...)`
- Cloned table has its own Time Travel (independent from source)

---

## Domain 3: Data Loading, Unloading & Connectivity (18%)

---

### 3.1 Stages — Complete Guide

**What is a Stage?**
A stage is a location where data files are stored (temporarily or permanently) for loading/unloading.

**Types of Stages:**

| Stage Type | Syntax | Scope | Location | Cloneable? |
|-----------|--------|-------|----------|------------|
| **User Stage** | `@~` | Per user | Snowflake-managed internal storage | NO |
| **Table Stage** | `@%table_name` | Per table | Snowflake-managed internal storage | NO |
| **Named Internal Stage** | `@stage_name` | Explicit object | Snowflake-managed internal storage | NO |
| **Named External Stage** | `@stage_name` | Explicit object | S3, Azure Blob, GCS | YES (metadata only) |

**User Stage (`@~`):**
- Every user has one automatically
- Cannot be altered or dropped
- Cannot set file format options on the stage itself
- Cannot be shared with other users
- No GRANT possible

**Table Stage (`@%table_name`):**
- Every table has one automatically
- Cannot be altered or dropped
- Cannot set file format options on the stage itself
- Data only loadable INTO that specific table
- No GRANT possible

**Named Internal Stage:**
- Created explicitly: `CREATE STAGE my_stage`
- Can have file format and copy options
- Can be granted to other roles
- Supports encryption (always encrypted)
- **Cannot be cloned**

**Named External Stage:**
- Points to cloud storage (S3/Azure/GCS)
- Created with URL + credentials: `CREATE STAGE my_ext_stage URL='s3://bucket/path/'`
- Uses Storage Integration for credential management
- Can be cloned (only metadata/definition, not the external data)

### 3.2 Directory Tables

**What:** A directory table is a built-in catalog of staged files within a stage. Enabling a directory table on a stage does not create a separate schema object (like a table) in the schema. We can query it using the stage name and the DIRECTORY() table function.

**Key Points:**
- Stores metadata about files in a stage (filename, size, MD5, last modified, etc.)
- Enabled on a stage: `ALTER STAGE my_stage SET DIRECTORY = (ENABLE = TRUE)`
- Must manually refresh: `ALTER STAGE my_stage REFRESH` (or auto-refresh for external stages)
- Query with: `SELECT * FROM DIRECTORY(@my_stage)`
- Useful for: listing files, tracking what's been staged, building pipelines
- Supports auto-refresh (event notifications from cloud storage) for external stages
- Works with internal and external stages

**Auto-refresh for Directory Tables:**
- External stages: uses cloud event notifications (SNS/SQS for AWS, Event Grid for Azure, Pub/Sub for GCS)
- Internal stages: must manually refresh

### 3.3 Data Loading Methods

| Method | Type | Best For | Warehouse Needed? |
|--------|------|----------|------------------|
| **COPY INTO** | Batch/Bulk | Large files, scheduled loads | YES |
| **Snowpipe** | Continuous/Auto | Real-time streaming, small files arriving continuously | NO (serverless) |
| **Snowpipe Streaming** | Real-time API | Lowest latency, row-level ingestion | NO (serverless) |
| **Web UI** | Manual | Small files (<50MB), one-time | YES |
| **INSERT** | SQL | Small amounts of data | YES |

**COPY INTO command:**
```sql
COPY INTO my_table
FROM @my_stage/path/
FILE_FORMAT = (TYPE = 'CSV' FIELD_DELIMITER = ',' SKIP_HEADER = 1)
ON_ERROR = 'CONTINUE'  -- or ABORT_STATEMENT, SKIP_FILE, SKIP_FILE_n
PATTERN = '.*\.csv'
FORCE = FALSE;  -- TRUE reloads already-loaded files
```

**Load History:**
- Snowflake tracks which files have been loaded (metadata)
- Prevents duplicate loading (64-day load history window)
- `FORCE = TRUE` bypasses this check
- `LOAD_HISTORY` view in INFORMATION_SCHEMA and ACCOUNT_USAGE

**Snowpipe:**
- Serverless, auto-ingest from stage
- Triggered by cloud event notifications OR REST API calls
- Uses a PIPE object: `CREATE PIPE ... AUTO_INGEST = TRUE AS COPY INTO ...`
- Billed per-second of compute used
- Cannot be used with `VALIDATION_MODE`

### 3.4 File Formats

| Structured | Semi-Structured |
|-----------|----------------|
| CSV, TSV | JSON |
| | Avro |
| | Parquet |
| | ORC |
| | XML |

**Semi-structured data** loaded into VARIANT column or directly into typed columns (with schema detection).

### 3.5 Data Unloading

- `COPY INTO @stage FROM table/query`
- Supported formats: CSV, JSON, Parquet
- Can unload to internal or external stage
- Single file: `SINGLE = TRUE` (may be slow for large data)
- Partitioned output: `PARTITION BY` clause

### 3.6 Connectors & Drivers

| Connector/Driver | Use Case |
|-----------------|----------|
| Snowflake Connector for Python | Python applications |
| Snowflake Connector for Spark | Apache Spark integration |
| Snowflake Connector for Kafka | Streaming from Kafka topics |
| JDBC Driver | Java applications |
| ODBC Driver | BI tools, general connectivity |
| Node.js Driver | JavaScript/Node applications |
| .NET Driver | .NET applications |
| Go Driver | Go applications |
| SnowSQL | CLI client |
| Snowsight | Web UI |

**Snowpark:**
- DataFrame API for Python, Java, Scala
- Code runs IN Snowflake (pushdown execution)
- No data movement outside Snowflake
- Supports UDFs, UDTFs, Stored Procedures

---

## Domain 4: Performance Optimization, Querying & Transformation (21%)

---

### 4.1 Virtual Warehouses

**Sizes:**

| Size | Servers | Credits/Hour |
|------|---------|-------------|
| X-Small | 1 | 1 |
| Small | 2 | 2 |
| Medium | 4 | 4 |
| Large | 8 | 8 |
| X-Large | 16 | 16 |
| 2X-Large | 32 | 32 |
| 3X-Large | 64 | 64 |
| 4X-Large | 128 | 128 |
| 5X-Large | 256 | 256 |
| 6X-Large | 512 | 512 |

**Each size UP = 2x the compute, 2x the credits.**

**Scaling Policy (Multi-Cluster Warehouses — Enterprise+):**

| Policy | Behavior |
|--------|----------|
| **Standard** | Adds clusters immediately when queries queue. Shuts down after 2-3 consecutive checks show no load. Favors performance. |
| **Economy** | Waits to add clusters until estimated 6+ minutes of work. Keeps clusters longer. Favors cost savings. |

**Multi-Cluster Warehouse:**
- MIN_CLUSTER_COUNT and MAX_CLUSTER_COUNT
- For concurrent query scaling (NOT faster single queries)
- Bigger warehouse size = faster single query; More clusters = more concurrent queries
- Resizing a Snowflake warehouse does not affect queries currently running; the change applies only to new queries.
- Multi-cluster warehouses are used to handle variable user demand

### 4.2 Types of Warehouses / Compute

| Type | Description | Key Points |
|------|-------------|------------|
| **Standard Warehouse** | Default, user-managed | You start/stop, pick size |
| **Multi-Cluster Warehouse** | Auto-scales clusters | Enterprise+, handles concurrency |
| **Snowpark-Optimized Warehouse** | Extra memory per node | For ML, large UDFs, memory-intensive Snowpark |
| **Serverless Compute** | Snowflake-managed, no warehouse needed | Used by: Snowpipe, Auto Clustering, Materialized View maintenance, Search Optimization, Replication, Tasks (serverless mode) |

**Serverless Features (no user warehouse needed):**
- Snowpipe
- Automatic Clustering
- Materialized View refresh
- Search Optimization Service maintenance
- Replication
- Serverless Tasks
- Query Acceleration Service

**Query Acceleration Service (QAS):**
- Offloads portions of query to serverless compute
- Best for queries with large scans + selective filters
- Enable: `ALTER WAREHOUSE SET ENABLE_QUERY_ACCELERATION = TRUE`
- Set max scale factor: `QUERY_ACCELERATION_MAX_SCALE_FACTOR`
- Not a replacement for sizing — works alongside warehouse

### 4.3 Query Profile

**What:** Visual execution plan of a query after it runs.

**Key metrics to look for:**

| Indicator | Meaning | Action |
|-----------|---------|--------|
| **Bytes Scanned** | How much data was read | High → improve pruning, add clustering key |
| **Percentage Scanned** | % of total table scanned | High → filter not pruning well |
| **Spillage to Local/Remote** | Data didn't fit in memory | Spilling to local disk (okay) or remote (bad) → upsize warehouse |
| **Exploding Joins** | Output rows >> input rows | Check join conditions |
| **Pruning** | Partitions scanned vs total | Low pruning % = good; High = need clustering |

**Spilling:**
- **Local spill** — data overflows to warehouse local SSD (some performance hit)
- **Remote spill** — overflows to remote storage (significant performance hit)
- Solution: Use a larger warehouse

**Query Profile Operators:**
- TableScan, Filter, Join (various types), Aggregate, Sort, Limit, WindowFunction, etc.

### 4.4 Query History

**Snowsight → Activity → Query History** (UI)

**SQL access:**
- `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` — account-wide, 365-day retention, 45-min latency
- `INFORMATION_SCHEMA.QUERY_HISTORY()` — table function, 7-day retention, real-time
- `INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER()` / `_BY_WAREHOUSE()`

**What you can see:**
- Query text, status, duration, bytes scanned, rows returned
- Warehouse used, user, role
- Compilation time, execution time
- Error messages for failed queries

### 4.5 Sampling Methods

**Purpose:** Return a random subset of rows from a table (useful for data exploration/testing).

**Syntax:**
```sql
SELECT * FROM my_table SAMPLE (10);          -- 10% of rows
SELECT * FROM my_table TABLESAMPLE (10);     -- Same (SAMPLE = TABLESAMPLE)
```

**Two Methods:**

| Method | Syntax | How It Works | Speed | Randomness |
|--------|--------|-------------|-------|------------|
| **Row-based (Bernoulli)** | `SAMPLE BERNOULLI (n)` or `SAMPLE ROW (n)` | Each row has n% probability of inclusion | Slower (row-by-row decision) | More random |
| **Block-based (System)** | `SAMPLE SYSTEM (n)` or `SAMPLE BLOCK (n)` | Each micro-partition has n% probability of inclusion | Faster (partition-level decision) | Less random (entire blocks in/out) |

**Key Points:**
- `n` is a percentage (e.g., 10 = 10%)
- Can also use fixed row count: `SAMPLE (100 ROWS)`
- SEED/REPEATABLE clause for reproducibility: `SAMPLE BERNOULLI (10) SEED(42)` (only with Bernoulli)
- Default method (if not specified) is Bernoulli/Row
- SYSTEM/BLOCK is faster for large tables but less uniform

### 4.6 Materialized Views

- Pre-computed query results stored and auto-maintained
- Enterprise+ feature
- Best for: frequent, expensive queries on data that changes infrequently
- Auto-refreshed by serverless compute (background)
- Restrictions: single table, no joins, no UDFs, limited aggregations
- Cost: storage + serverless maintenance credits

### 4.7 Search Optimization Service

- Enterprise+ feature
- Speeds up point lookups and equality/IN predicates
- Background serverless process maintains search access paths
- Enable: `ALTER TABLE t ADD SEARCH OPTIMIZATION`
- Best for: highly selective queries on columns without clustering benefit
- Cost: serverless compute for maintenance + storage

### 4.8 Query Constructs & Transformations

**Common Table Expressions (CTEs):**
```sql
WITH cte AS (
  SELECT ... FROM ...
)
SELECT * FROM cte;
```

**Window Functions:**
```sql
SELECT col, ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY sort_col) as rn
FROM table;
```

**Semi-Structured Data Querying:**
```sql
-- Dot notation
SELECT v:name::STRING, v:address.city::STRING FROM json_table;

-- LATERAL FLATTEN (for arrays)
SELECT t.id, f.value::STRING as item
FROM my_table t, LATERAL FLATTEN(input => t.items_array) f;
```

**Stored Procedures:**
- Written in SQL, JavaScript, Python, Java, Scala
- Can perform DDL/DML
- `EXECUTE AS OWNER` (default) or `EXECUTE AS CALLER`

**UDFs (User-Defined Functions):**
- Written in SQL, JavaScript, Python, Java, Scala
- Scalar (return 1 value per row) or Tabular (UDTF — return table)
- Cannot perform DML (read-only)
- Immutable or Volatile

**Streams:**
- Capture CDC (Change Data Capture) on tables
- Track INSERT, UPDATE, DELETE
- Types: Standard, Append-only, Insert-only
- Used with Tasks for continuous pipelines

**Tasks:**
- Schedule SQL execution (cron or interval)
- Can be chained (DAG/tree structure)
- Can use serverless compute or a warehouse
- Triggered by schedule OR WHEN condition (e.g., `SYSTEM$STREAM_HAS_DATA()`)

**Dynamic Tables:**
- Declarative data transformation
- Define target as a query; Snowflake keeps it fresh
- `TARGET_LAG` — how fresh the data should be
- Replaces streams + tasks for many ETL patterns

---

## Domain 5: Data Collaboration (10%)

---

### 5.1 Secure Data Sharing — Complete Guide

**What:** Share live, read-only data between Snowflake accounts with ZERO data copying.

**Key Concepts:**

| Term | Definition |
|------|------------|
| **Provider** | Account that shares data (creates the share) |
| **Consumer** | Account that accesses shared data |
| **Share** | Named object containing grants to databases/schemas/objects |
| **Imported Database** | Consumer creates a database from a share |

**How it works:**
1. Provider creates a SHARE: `CREATE SHARE my_share;`
2. Provider grants access to objects:
   ```sql
   GRANT USAGE ON DATABASE db TO SHARE my_share;
   GRANT USAGE ON SCHEMA db.schema TO SHARE my_share;
   GRANT SELECT ON TABLE db.schema.table TO SHARE my_share;
   ```
3. Provider adds consumer account: `ALTER SHARE my_share ADD ACCOUNTS = consumer_org.consumer_account;`
4. Consumer creates database from share: `CREATE DATABASE shared_db FROM SHARE provider_org.provider_account.my_share;`

**Key Points:**
- Data is NOT copied — consumer reads provider's storage (zero-copy)
- Provider pays for storage; Consumer pays for compute (their warehouse)
- Shared data is **read-only** for consumers
- Consumer can share the data further? **NO** — cannot re-share
- Changes by provider are immediately visible to consumer
- Cross-region/cross-cloud sharing requires **Replication**
- Can share: tables, external tables, secure views, secure UDFs, secure materialized views
- **Cannot share:** non-secure views, stages, pipes, tasks, streams

**Secure Views for Sharing:**
- Always use SECURE views when sharing (hides view definition from consumer)
- `CREATE SECURE VIEW ...`
- Query optimizer may be limited with secure views (can't push predicates through)

### 5.2 Reader Accounts (Managed Accounts)

- For sharing with non-Snowflake customers
- Provider creates and manages reader accounts
- Provider pays for BOTH storage AND compute
- Limited functionality (no loading data, minimal admin)
- `CREATE MANAGED ACCOUNT reader1 ADMIN_NAME='...' ADMIN_PASSWORD='...'`

### 5.3 Snowflake Marketplace

- Public marketplace for data products
- **Listings:** providers publish data products
- **Personalized listings:** private, targeted to specific consumers
- Types: Standard (free), Paid, Personalized
- Data products: live data shares, sample datasets, data apps (Native Apps)
- No ETL needed — instant access once you "get" a listing

### 5.4 Data Exchange

- Private marketplace for a group of accounts
- Invitation-based
- Organization controls membership
- Members can be providers and/or consumers

### 5.5 Replication & Failover

**Replication:**
- Copy databases/shares across regions or cloud platforms
- Primary → Secondary (read-only replica)
- Can replicate: databases, shares, account objects (users, roles, warehouses)

**Failover (Business Critical+):**
- Promote secondary to primary in disaster recovery
- `ALTER DATABASE ... PRIMARY` on secondary
- Client redirect: automatic redirect to failover account

---

## Quick Reference: Exam Traps & Common Confusions

---

| Topic | Key Fact to Remember |
|-------|---------------------|
| Result Cache | 24 hrs, same query + same role + data unchanged. **No warehouse credits.** |
| ACCOUNTADMIN | Don't use as default role. Always enable MFA. |
| Transient Tables | Max 1 day Time Travel, NO Fail-Safe, any edition |
| Temporary Tables | Session-scoped only, max 1 day TT, NO Fail-Safe |
| COPY INTO loads | 64-day metadata tracking prevents re-loading |
| Snowpipe | Serverless, AUTO_INGEST, uses PIPE object |
| Clustering Keys | Only for very large tables (TB+), not a replacement for indexes |
| Internal Stage Cloning | CANNOT clone internal stages (named, user, or table) |
| External Stage Cloning | CAN clone (metadata only) |
| Clone Privileges | Single object clone = no privileges copied. Container clone (DB/schema) = privileges copied |
| VARIANT max size | 16 MB per row |
| Scaling UP | Bigger warehouse → faster complex queries |
| Scaling OUT | More clusters → more concurrent queries |
| Serverless Tasks | `CREATE TASK ... SERVERLESS = TRUE` (no warehouse specified) |
| Secure Views | Definition hidden, required for sharing, optimizer limitations |
| Tri-Secret Secure | Customer key + Snowflake key = composite key. Business Critical+ only |
| Re-keying vs Rotation | Rotation = new key for new data. Re-keying = old data encrypted with new key |
| INFORMATION_SCHEMA | Real-time, per-database, accessible by any role with DB access |
| ACCOUNT_USAGE | 45 min delay, account-wide, ACCOUNTADMIN (or granted IMPORTED PRIVILEGES) |
| DATA_RETENTION_TIME_IN_DAYS | Standard = max 1 day. Enterprise+ = max 90 days |
| Streams on Shared Tables | NOT supported (can't create stream on shared/imported table) |
| Zero-copy sharing | Provider pays storage, Consumer pays compute |
| Reader Accounts | Provider pays everything (storage + compute) |
| UNDROP | Works within Time Travel period. UNDROP TABLE, SCHEMA, DATABASE |
| Directory Tables | Built-in file catalog on stages. Must refresh manually for internal stages. |
| Sampling | BERNOULLI = row-level (slower, more random). SYSTEM = block-level (faster, less random) |
| Query Acceleration | Serverless add-on to warehouse. Helps with large scans + selective filters. |
| Dynamic Tables | Declarative ETL. TARGET_LAG controls freshness. Replaces streams+tasks pattern. |
| Cortex Functions | Run inside Snowflake. Serverless billing. Data never leaves account. |

## Clustering in Snowflake

---

### 1. What Is Clustering?

Clustering is a technique where Snowflake **co-locates rows with similar values into the same micro-partitions**, enabling better partition pruning during queries — which improves performance and reduces scan cost.

---

### 2. How to Choose Clustering Columns

| Priority | Guideline |
|----------|----------|
| 1st | Columns most frequently used in **WHERE** clauses |
| 2nd | Columns used in **JOIN** predicates |
| 3rd | Balance cardinality: too low → minimal pruning, too high → inefficient grouping |
| 4th | For multi-column clustering keys: order from **low to high** cardinality |

---

### 3. Clustering Metadata (Snowflake tracks automatically)

| Metric | Meaning |
|--------|--------|
| Total micro-partitions | Number of micro-partitions in the table |
| Overlapping micro-partitions | Count of micro-partitions whose value ranges overlap with at least one other |
| Overlap depth | Maximum number of micro-partitions that must be scanned in the worst case for a single value lookup |

---

### 4. Overlapping Micro-Partitions & Overlap Depth — Explained

- **Overlapping micro-partition:** A micro-partition whose value range overlaps with at least one other micro-partition.
- **Overlap depth:** The maximum number of micro-partitions that share a common value range at any point — i.e., the worst-case number of partitions to scan for a single value.

---

### 5. Visual Examples

#### Example A: Worst clustering (completely unsorted)

```
Micro-Partition    Range       
─────────────────────────────
  MP1              [A ─── Z]  
  MP2              [A ─── Z]       Total MP: 5
  MP3              [A ─── Z]       Overlapping MP: 5
  MP4              [A ─── Z]       Overlap Depth: 5
  MP5              [A ─── Z]  
```
**Why?** Every MP spans the full range. For ANY value (e.g., 'M'), Snowflake must scan all 5 MPs.

---

#### Example B: Partially clustered

```
Micro-Partition    Range       
─────────────────────────────
  MP1              [L ─── Z]  ←┐
  MP2              [L ─── Z]  ←┤ These 3 overlap with each other
  MP3              [L ─── Z]  ←┘
  MP4              [A ─── E]       Total MP: 5
  MP5              [F ─── K]       Overlapping MP: 3 (MP1, MP2, MP3)
                                   Overlap Depth: 3
```
**Why depth = 3?** For value 'M', you must scan MP1 + MP2 + MP3 (all contain 'M' in their range). MP4 and MP5 are pruned.

---

#### Example C: Better clustering (some overlap remains)

```
Micro-Partition    Range       
─────────────────────────────
  MP1              [A ─── D]  
  MP2              [E ─── J]       Total MP: 5
  MP3              [K ─── N]  ←┐   Overlapping MP: 3 (MP3, MP4, MP5)
  MP4              [L ─── S]  ←┤   Overlap Depth: 2
  MP5              [Q ─── Z]  ←┘
```
**Why depth = 2?** The worst case is a value like 'L' which falls in both MP3 [K-N] and MP4 [L-S] — at most 2 MPs to scan. Similarly 'Q' falls in MP4 and MP5.

**Why overlapping = 3?** MP3, MP4, and MP5 are each involved in at least one overlap (MP3∩MP4 at K-N/L-S, MP4∩MP5 at L-S/Q-Z). MP1 and MP2 have no overlaps.

---

#### Example D: Perfect clustering (no overlap)

```
Micro-Partition    Range       
─────────────────────────────
  MP1              [A ─── D]  
  MP2              [E ─── J]       Total MP: 5
  MP3              [K ─── N]       Overlapping MP: 0
  MP4              [O ─── S]       Overlap Depth: 1
  MP5              [T ─── Z]  
```
**Why depth = 1?** Every value exists in exactly one MP. Any query on a single value scans at most 1 micro-partition. This is the ideal state.

---

### 6. Clustering Depth

**Clustering Depth = average overlap depth across the entire table.**

```sql
-- Check clustering depth
SELECT SYSTEM$CLUSTERING_DEPTH('my_table');

-- Check clustering depth on specific columns
SELECT SYSTEM$CLUSTERING_DEPTH('my_table', '(col1, col2)');

-- Detailed clustering info
SELECT SYSTEM$CLUSTERING_INFORMATION('my_table');
```

| Depth Value | Meaning |
|-------------|--------|
| 1 | Perfectly clustered (no overlap) |
| 2-3 | Well clustered |
| High (e.g., 10+) | Poorly clustered — consider adding a clustering key |

---

### 7. When to Use Clustering Depth

| Purpose | How It Helps |
|---------|-------------|
| Decide if a table needs a clustering key | High depth on frequently filtered columns → add key |
| Estimate query performance | Lower depth → better pruning → faster queries |
| Monitor clustering health over time | Rising depth on large tables → reclustering may be needed |

---

**Q. What does a clustering depth of 1 mean?**
> **Ans:** It means zero overlap — each value exists in exactly one micro-partition. Snowflake can prune all other partitions, giving optimal scan performance.

**Q. Does Snowflake automatically recluster?**
> **Ans:** Yes. Once a clustering key is defined, Snowflake's **Automatic Clustering** service (serverless) maintains clustering in the background as DML changes the table. It is billed as serverless credits.

## Query Acceleration Service (QAS)

---

### What is it?

Query Acceleration Service (QAS) is a serverless feature that accelerates eligible queries by offloading **portions of a query** to shared serverless compute resources provided by Snowflake. It does NOT replace your warehouse — it **supplements** it.

**Think of it this way:**
- Your warehouse handles the main query execution
- QAS takes the "heavy scanning" parts and runs them on additional serverless nodes in parallel
- Result: faster query, same warehouse size

```
Your Warehouse + Additional Serverless Compute → Faster Query Execution
```

---

### Does it optimize the QUERY or the WAREHOUSE?

**Neither exactly — it accelerates specific query workloads by adding temporary serverless compute.**

- It does NOT rewrite your query (not a query optimizer)
- It does NOT resize your warehouse
- It ADDS extra compute power (serverless) for the scan-heavy portions of eligible queries
- Your warehouse still does the final processing (joins, aggregations on filtered data)

**Performance improvement mechanism:**
```
Without QAS:  Warehouse scans 1 billion rows alone         → 60 seconds
With QAS:     Warehouse + serverless nodes scan in parallel → 15 seconds
```

**QAS vs Result Cache:**
> Result Cache avoids executing a query by reusing a previously computed result, whereas Query Acceleration Service speeds up query execution when the result cannot be reused and the query must be run again.

---

### Which Queries Are Eligible?

QAS works best for queries that:
1. **Scan a LOT of data** but **return a SMALL result** (selective filters on large tables)
2. Are **outlier queries** — most queries are fast, but some are disproportionately slow
3. Perform aggregations over large data with filters

**NOT eligible / won't benefit:**

| Pattern | Reason |
|---------|--------|
| Queries already fast (< few seconds) | Nothing to accelerate |
| Queries that return most of the table | Full scan needed anyway |
| Queries blocked by joins/sorting (not scanning) | Bottleneck isn't the scan |
| INSERT/COPY/DML operations | Only SELECT queries benefit |

---

### The Scale Factor

**Scale Factor** = how much additional serverless compute QAS can use (multiplier of your warehouse size).

| Scale Factor | Meaning |
|-------------|--------|
| **0** | QAS is disabled |
| **1** | QAS can use up to 1x your warehouse compute additionally |
| **8** (default max) | QAS can use up to 8x your warehouse compute |
| **16** | Maximum allowed |

---

### How to Identify if QAS Will Help (Step-by-Step)

**Step 1: Find eligible queries from previously executed queries**
```sql
SELECT query_id, eligible_query_acceleration_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ACCELERATION_ELIGIBLE
ORDER BY eligible_query_acceleration_time DESC;
```

**Step 2: Get the recommended scale factor for an eligible query**
```sql
SELECT PARSE_JSON(SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>'));
```

**Step 3: Find the eligible query details by query_id**
```sql
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_ID = '<query_id>';
```

**Step 4: Disable query cache (so re-run actually executes)**
```sql
ALTER SESSION SET USE_CACHED_RESULT = FALSE;
```

**Step 5: Enable QAS on the warehouse**
```sql
ALTER WAREHOUSE MY_WAREHOUSE
  SET ENABLE_QUERY_ACCELERATION = TRUE
      QUERY_ACCELERATION_MAX_SCALE_FACTOR = <scale_factor>;
```

---

### Monitoring QAS Credit Usage After Enabling

**View aggregate QAS credit consumption for a warehouse (last 7 days):**

This shows warehouse-level totals — credits consumed, files scanned, and bytes scanned by QAS. It does NOT show individual query IDs.

| Column | Description |
|--------|-------------|
| START_TIME | Start of the interval |
| END_TIME | End of the interval |
| CREDITS_USED | Serverless credits consumed by QAS |
| NUM_FILES_SCANNED | Number of files scanned by QAS |
| NUM_BYTES_SCANNED | Bytes scanned by QAS |
| WAREHOUSE_NAME | Which warehouse used QAS |

```sql
SELECT * FROM TABLE(
  INFORMATION_SCHEMA.QUERY_ACCELERATION_HISTORY(
    DATE_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP()),
    DATE_RANGE_END => CURRENT_TIMESTAMP(),
    WAREHOUSE_NAME => 'MY_WAREHOUSE'
  )
);
```

**Find individual queries that were accelerated by QAS:**

Use QUERY_HISTORY to see per-query QAS activity (this one HAS query_id):
```sql
SELECT query_id, query_acceleration_bytes_scanned, total_elapsed_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE warehouse_name = 'MY_WAREHOUSE'
  AND query_acceleration_bytes_scanned > 0
ORDER BY query_acceleration_bytes_scanned DESC;
```

---

### How to Enable

**Estimate acceleration potential for a warehouse (before enabling):**
```sql
SELECT SYSTEM$ESTIMATE_QUERY_ACCELERATION('MY_WAREHOUSE');
```

```sql
ALTER WAREHOUSE MY_WAREHOUSE
  SET ENABLE_QUERY_ACCELERATION = TRUE
      QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;
```

- `QUERY_ACCELERATION_MAX_SCALE_FACTOR` limits how much serverless compute can be used
- Higher factor = potentially faster but more expensive
- Set to 0 = disabled

---

### Cost

- Billed as **serverless credits** (separate from warehouse credits)
- You only pay when QAS actually accelerates a query
- The scale factor caps your maximum spend per query
- No cost if no queries are accelerated

---

### Summary for Exam

| Question | Answer |
|----------|--------|
| What does QAS do? | Adds serverless compute to speed up scan-heavy queries |
| Does it replace the warehouse? | NO — supplements it |
| What queries benefit? | Large scans with selective filters (few rows returned) |
| What's the scale factor? | Max multiplier of additional serverless compute (0-16) |
| How is it billed? | Serverless credits (pay only when used) |
| Which edition? | Enterprise+ |
| How to enable? | `ALTER WAREHOUSE ... SET ENABLE_QUERY_ACCELERATION = TRUE` |
| Does it help DML/COPY? | NO — only SELECT queries |

## Search Optimization Service (SOS)

---

### What is it?

Search Optimization Service is a background serverless feature that creates and maintains **search access paths** — an optimized data structure that allows Snowflake to quickly locate rows matching point lookups and selective filters WITHOUT scanning all micro-partitions.

```
Normal query:    Scan many micro-partitions → find matching rows → slow
With SOS:        Use search access paths → jump directly to relevant partitions → fast
```

---

### How Does it Work?

1. You enable SOS on a table (or specific columns)
2. A **background serverless process** builds and maintains **search access paths**
3. These paths are like a secondary index — they map values to micro-partitions
4. When a query has a selective filter (e.g., `WHERE id = 12345`), Snowflake uses the search access path to skip irrelevant partitions entirely
5. The maintenance service keeps paths updated as data changes (INSERT/UPDATE/DELETE)

**Key difference from clustering:**
- Clustering physically reorganizes data → helps range scans
- SOS creates a lookup structure without moving data → helps point lookups

---

### Is There Latency?

**YES — there is a build-up latency:**
- After enabling, it takes time to build the search access paths (minutes to hours depending on table size)
- The background service must process all existing micro-partitions
- New data also needs time to be indexed by the maintenance service
- You won't see performance improvement immediately after enabling

**No latency at query time** — once paths are built, queries are accelerated instantly.

---

### Accessing SOS metadata for any table
- Use show command to get metadata of table
  ```sql
  SHOW TABLES LIKE '%<table_name>%'
  ```
- Look for below 3 columns:
  a) search_optimization : Specifies whether SOS enabled or not
  b) search_optimization_progress : Shows progress of SOS for the table [if value is 100, progress is completed]
  c) search_optimization_bytes : Shows bytes in SOS

---

### Does it Cost Extra Storage?

**YES — SOS has two costs:**

| Cost Type | Description |
|-----------|-------------|
| **Storage** | Search access paths consume additional storage (stored alongside your data) |
| **Serverless Compute** | Background maintenance service uses serverless credits to build and keep paths updated |

You pay for:
1. **Storage** — for the search access path data structure
2. **Compute credits** — serverless credits for the maintenance service that builds and refreshes paths

---

### Which Queries Benefit?

| Query Pattern | Example | Benefits? |
|--------------|---------|----------|
| Equality predicates | `WHERE user_id = 500` | YES |
| IN list | `WHERE status IN ('A', 'B', 'C')` | YES |
| LIKE with substring | `WHERE name LIKE '%smith%'` | YES |
| Geospatial functions | `WHERE ST_CONTAINS(geo, point)` | YES |
| VARIANT field access | `WHERE v:key = 'value'` | YES |
| Range scans (large) | `WHERE date BETWEEN '2024-01-01' AND '2024-12-31'` | NO — use clustering instead |
| Full table scans | `SELECT *` with no filter | NO |

**Best for:** Highly selective queries on large tables where clustering doesn't help (e.g., lookup by ID, substring search).

---

### How to Enable

**Enable on entire table (all columns):**
```sql
ALTER TABLE my_table ADD SEARCH OPTIMIZATION;
```

**Enable on specific columns (recommended for cost control):**
```sql
ALTER TABLE my_table ADD SEARCH OPTIMIZATION
  ON EQUALITY(user_id, email);
```

**Enable for substring searches:**
```sql
ALTER TABLE my_table ADD SEARCH OPTIMIZATION
  ON SUBSTRING(name, address);
```

**Enable for geospatial:**
```sql
ALTER TABLE my_table ADD SEARCH OPTIMIZATION
  ON GEO(geo_column);
```

**Check optimization status:**
```sql
-- See if search optimization is active and what % is built
SELECT * FROM TABLE(INFORMATION_SCHEMA.SEARCH_OPTIMIZATION_HISTORY(
  DATE_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP()),
  DATE_RANGE_END => CURRENT_TIMESTAMP(),
  TABLE_NAME => 'MY_DB.MY_SCHEMA.MY_TABLE'
));
```

**Describe what's optimized:**
```sql
DESCRIBE SEARCH OPTIMIZATION ON my_table;
```

**Remove search optimization:**
```sql
ALTER TABLE my_table DROP SEARCH OPTIMIZATION;
```

---

### SOS vs Clustering vs QAS

| Feature | Search Optimization | Clustering | Query Acceleration |
|---------|-------------------|------------|-------------------|
| Best for | Point lookups, substring, IN lists | Range filters, ordered scans | Outlier scan-heavy queries |
| How it works | Builds search access paths (like an index) | Physically reorders data | Adds extra serverless compute |
| Storage cost | YES | YES (re-clustering) | NO |
| Compute cost | Serverless (maintenance) | Serverless (auto-clustering) | Serverless (per-query) |
| Edition | Enterprise+ | Enterprise+ (auto-clustering) | Enterprise+ |
| Latency to take effect | Minutes to hours (build time) | Hours (re-clustering) | Immediate (per-query) |

---

### Summary for Exam

| Question | Answer |
|----------|--------|
| What does SOS do? | Creates search access paths for fast point lookups |
| Does it move/reorder data? | NO — builds a separate lookup structure |
| Extra storage? | YES — search access paths need storage |
| Extra compute? | YES — serverless credits for background maintenance |
| Latency after enabling? | YES — takes time to build paths (not instant) |
| Which edition? | Enterprise+ |
| How to enable? | `ALTER TABLE ... ADD SEARCH OPTIMIZATION` |
| Best query types? | Equality, IN, LIKE substring, geospatial, VARIANT fields |

## Snowflake Network Policies

---

### Purpose

Network Policies control **who can connect** to Snowflake based on source IP addresses. They are used for **connection/login control**, not object-level permissions.

---

### CIDR Notation

| CIDR | Meaning | IPs Covered |
|------|---------|-------------|
| `192.168.1.10/32` | Single IP | 1 |
| `192.168.1.0/24` | Class C subnet | 256 |
| `10.0.0.0/8` | Large private network | ~16.7 million |
| `0.0.0.0/0` | ALL IPv4 addresses | Every IP |

---

### Allowlist vs Blocklist

**Allowlist (ALLOWED_IP_LIST):**
```sql
CREATE NETWORK POLICY my_policy
  ALLOWED_IP_LIST = ('203.0.113.0/24');
```
| Source IP | Result |
|-----------|--------|
| In the list | Allowed |
| NOT in the list | Blocked (automatically) |

> Snowflake implicitly blocks everything not in the allowlist. You don't need a blocklist.

**Blocklist (BLOCKED_IP_LIST):**
```sql
CREATE NETWORK POLICY my_policy
  ALLOWED_IP_LIST = ('0.0.0.0/0')
  BLOCKED_IP_LIST = ('203.0.113.0/24');
```
| Source IP | Result |
|-----------|--------|
| In the blocked list | Blocked |
| All other IPs | Allowed |

> **Important:** `BLOCKED_IP_LIST` only works when used WITH an `ALLOWED_IP_LIST`. You cannot create a policy with only a blocklist — you must allow something first.

---

### Why `BLOCKED_IP_LIST = ('0.0.0.0/0')` is Dangerous

```
BLOCKED_IP_LIST = ('0.0.0.0/0')  →  Block EVERY IPv4 address  →  Nobody can connect
```

This locks out:
- ACCOUNTADMIN
- SECURITYADMIN
- SYSADMIN
- Service Accounts
- Snowsight Users
- SnowSQL Users
- **Everyone. No exceptions.**

---

### Login Flow — Where Network Policy Sits

```
Client (Laptop / App)
       ↓
  Network Policy Check    ← IP checked HERE (first gate)
       ↓
  Authentication          ← Username/Password, SSO, MFA
       ↓
  Role Authorization      ← ACCOUNTADMIN, SYSADMIN, etc.
       ↓
  Access Granted          ← Database, Schema, Table access
```

**If Network Policy blocks your IP:**
```
Client
   ↓
Network Policy  →  ❌ REJECTED (connection refused)

   Authentication?   Never reached.
   Role validation?  Never reached.
```

---

### ACCOUNTADMIN Cannot Bypass Network Policies

| Myth | Reality |
|------|--------|
| ACCOUNTADMIN can bypass Network Policies | **WRONG** — Network Policy is evaluated BEFORE role assignment |

The role doesn't matter because the IP check happens at the connection layer, before Snowflake even knows which role you want to use.

---

### Recovery if Everyone is Locked Out

| Scenario | Recovery Option |
|----------|----------------|
| Another admin has a **user-level** network policy that still allows their IP | That admin can log in and fix the account-level policy |
| No one can log in at all | Contact **Snowflake Support** — they can remove the policy |

> **User-level vs Account-level:** A user-level network policy overrides the account-level policy for that specific user. This is why it can serve as a safety net.

---

### Where Network Policies Can Be Applied

| Level | Effect | SQL |
|-------|--------|-----|
| **Account** | Applies to all users in the account | `ALTER ACCOUNT SET NETWORK_POLICY = 'my_policy'` |
| **User** | Applies to a specific user (overrides account-level) | `ALTER USER john SET NETWORK_POLICY = 'my_policy'` |
| **Security Integration** | Applies to SSO/OAuth connections | Set on the integration object |

**Priority:** User-level policy > Account-level policy (user-level wins if both exist)

---

### Best Practices

1. **Use ALLOWED_IP_LIST** to restrict access — safer than blocking
2. **Never set** `BLOCKED_IP_LIST = ('0.0.0.0/0')` at account level
3. **Always keep a backup admin** with a user-level policy allowing their IP
4. **Test policies on a single user first** before applying to the account
5. Only SECURITYADMIN or ACCOUNTADMIN can create and assign network policies

---

### Exam Nuggets

| Point | Remember |
|-------|----------|
| Network Policy = | Connection control (IP-based) |
| Roles = | Authorization control (object-based) |
| Evaluation order | Network Policy → Authentication → Role → Access |
| `0.0.0.0/0` means | ALL IPv4 addresses |
| Blocked IP = | Cannot log in at all |
| ACCOUNTADMIN bypass? | NO — policy is checked before roles |
| User-level vs Account-level | User-level overrides account-level |
| Who can create policies? | SECURITYADMIN or ACCOUNTADMIN |
| Prefer | `ALLOWED_IP_LIST` over blocking `0.0.0.0/0` |

## Snowpipe and Its Types

Snowflake provides two continuous ingestion services:

---

### 1. Snowpipe (Classic)

- Automatically loads **files** as they arrive in a stage
- Uses cloud event notifications (S3 SQS, Azure Event Grid, GCS Pub/Sub) or REST API
- Data is loaded in micro-batches
- Latency: seconds to minutes (near real-time)

```
Source File → Stage → Snowpipe → Target Table
```

### 2. Snowpipe Streaming

- Sends **rows directly** to Snowflake — no files, no staging
- Applications use the Snowpipe Streaming SDK (Java) or Kafka connector
- Latency: sub-second to seconds (true real-time)
- Ideal for IoT, clickstreams, telemetry, CDC

```
Application → Snowpipe Streaming API → Target Table
```

---

### COPY INTO vs Snowpipe vs Snowpipe Streaming

| Feature | COPY INTO | Snowpipe | Snowpipe Streaming |
|---------|-----------|----------|-------------------|
| Loading type | Batch | Continuous (file-based) | Real-time (row-based) |
| Trigger | Manual / Scheduled | Automatic (event notification) | Continuous (SDK/Kafka) |
| Uses files | Yes | Yes | No |
| Requires stage | Yes | Yes | No |
| Latency | Minutes to hours | Seconds to minutes | Milliseconds to seconds |
| Compute | User warehouse | Serverless | Serverless |
| Cost | Warehouse credits | Snowpipe credits (per-file) | Streaming credits |
| Duplicate prevention | Manual (64-day metadata) | Automatic file tracking (14 days) | Managed by streaming service |
| Data source | Files | Files | Applications, Kafka, IoT, CDC |
| Best for | Large scheduled batch loads | Continuous file arrival | High-frequency event streams |

---

### Quick Memory Trick

| Service | One-Word Summary | Model |
|---------|-----------------|-------|
| COPY INTO | **Batch** | File-based, manual |
| Snowpipe | **Near real-time** | File-based, automatic |
| Snowpipe Streaming | **Real-time** | Row-based, direct |

---

**Q. When should you use Snowpipe Streaming instead of Snowpipe?**

> **Ans:** Use **Snowpipe Streaming** when you need low-latency, real-time ingestion and want to send records directly into Snowflake without creating files in a stage. Use **Snowpipe** when data arrives as files and near real-time ingestion (seconds to minutes) is sufficient.

## VALIDATION_MODE vs VALIDATE

---

### VALIDATION_MODE

Used **with COPY INTO** to check data files for errors **without loading** the data. It reads the file, validates it, shows errors, but does not insert any rows.

**Use case:** Pre-load validation — catch issues before committing data.

**Possible Modes:**

| Mode | Purpose |
|------|--------|
| `RETURN_ERRORS` | Return only the rows that have errors |
| `RETURN_ALL_ERRORS` | Return all errors across all files |
| `RETURN_n_ROWS` | Preview first *n* rows (e.g., `RETURN_5_ROWS`) |

**Example:**
```sql
COPY INTO employee
FROM @my_stage
FILE_FORMAT = (TYPE = CSV)
VALIDATION_MODE = RETURN_ERRORS;
```

---

### VALIDATE()

A **table function** used **after** a COPY command to view load errors from a previous load operation.

**Use case:** Post-load analysis — review what went wrong in a completed COPY.

- Analyzes previous COPY results
- Returns rejected rows and error details
- Used after load execution

**Example:**
```sql
SELECT *
FROM TABLE(
  VALIDATE(employee, JOB_ID => '_last')
);
```

> `JOB_ID => '_last'` refers to the most recent COPY operation on that table.

---

### Quick Comparison

| | VALIDATION_MODE | VALIDATE() |
|--|----------------|------------|
| **When** | Before/during COPY (no data loaded) | After COPY (data already loaded) |
| **Type** | COPY INTO parameter | Table function |
| **Purpose** | Preview & check files for errors | Review errors from a past load |
| **Loads data?** | NO | N/A (load already happened) |

---

**Exam One-Liner:**
- `VALIDATION_MODE` = Validate files **before** loading data
- `VALIDATE()` = Review load errors **after** a COPY operation

## Streams and Their Types in Snowflake

Streams are Snowflake's mechanism for **Change Data Capture (CDC)**. A stream tracks DML changes (INSERT, UPDATE, DELETE) made to a source object since the last time the stream was consumed.

---

### Stream Types

#### 1. Standard Stream (default)

- Tracks **all** DML changes: INSERT, UPDATE, DELETE
- Uses hidden metadata columns to identify row changes
- Suitable when you need complete change tracking

```sql
CREATE STREAM emp_stream ON TABLE employees;
```

#### 2. Append-Only Stream

- Tracks only **INSERT** operations
- Does NOT record UPDATE or DELETE
- More efficient than standard stream for append-only workloads

```sql
CREATE STREAM sales_stream ON TABLE sales APPEND_ONLY = TRUE;
```

**Use cases:** Log ingestion, event data, transaction records where only new rows are added.

#### 3. Insert-Only Stream

- Used **only on external tables**
- Tracks only new file data appearing in external storage
- Does NOT track deletions or updates

```sql
CREATE STREAM ext_stream ON EXTERNAL TABLE ext_sales INSERT_ONLY = TRUE;
```

**Use cases:** Data ingestion from S3, Azure Blob Storage, or GCS through external tables.

---

### Comparison

| Feature | Standard | Append-Only | Insert-Only |
|---------|----------|-------------|-------------|
| INSERT tracking | Yes | Yes | Yes |
| UPDATE tracking | Yes | No | No |
| DELETE tracking | Yes | No | No |
| Works on tables | Yes | Yes | No |
| Works on views | Yes | Yes | No |
| Works on external tables | No | No | Yes |
| Best for | Full CDC | Append-only data | External data ingestion |

---

### Stream Metadata Columns

When querying a stream, Snowflake exposes three metadata columns:

| Column | Type | Meaning |
|--------|------|--------|
| `METADATA$ACTION` | VARCHAR | `'INSERT'` or `'DELETE'` |
| `METADATA$ISUPDATE` | BOOLEAN | `TRUE` if the row is part of an UPDATE |
| `METADATA$ROW_ID` | VARCHAR | Unique identifier for the row |

**How UPDATEs appear:** An UPDATE produces two rows — a DELETE (old values) + an INSERT (new values), both with `METADATA$ISUPDATE = TRUE`.

```sql
SELECT * FROM emp_stream;
-- Shows METADATA$ACTION, METADATA$ISUPDATE, METADATA$ROW_ID alongside data columns
```

---

**Q. What are the types of streams in Snowflake?**

> **Ans:** Snowflake supports three types of streams:
> 1. **Standard** — Tracks INSERT, UPDATE, and DELETE operations.
> 2. **Append-Only** — Tracks only INSERTs; ignores updates/deletes.
> 3. **Insert-Only** — Used exclusively with external tables; tracks only newly added records.
>
> Streams are commonly used for CDC and incremental ETL/ELT processing.

### Row-Level Security & Column-Level Security

> Available in **Enterprise edition and higher** only.

---

#### Row-Level Security (Row Access Policies)

A Row Access Policy filters **which rows** a user can see at query time.

```sql
CREATE ROW ACCESS POLICY region_policy AS
  (region_val VARCHAR) RETURNS BOOLEAN ->
  CURRENT_ROLE() = 'ADMIN'
  OR region_val IN (
    SELECT region FROM role_region_map WHERE role_name = CURRENT_ROLE()
  );

-- Apply to table
ALTER TABLE sales ADD ROW ACCESS POLICY region_policy ON (region);
```

**Key Facts:**

| Aspect | Detail |
|--------|--------|
| Evaluation | Dynamic at query runtime (not at load time) |
| Returns | BOOLEAN — `TRUE` = row visible, `FALSE` = row hidden |
| Per table/view | Only **ONE** row access policy per object |
| Multi-object | Same policy can be applied to multiple tables/views |
| Complex logic | Must combine all conditions in one policy (AND/OR) |
| ACCOUNTADMIN bypass | **No** — ACCOUNTADMIN does NOT automatically bypass (must be coded in policy) |
| Policy owner bypass | **No** — OWNERSHIP on the policy does NOT grant bypass |

**Context functions commonly used in policies:**

| Function | Filters Based On |
|----------|------------------|
| `CURRENT_ROLE()` | User's active role |
| `CURRENT_USER()` | Username |
| `CURRENT_ACCOUNT()` | Account identifier |
| `IS_ROLE_IN_SESSION()` | Whether a specific role is available in the session |

---

#### Column-Level Security (Dynamic Data Masking)

A Masking Policy controls **what value** a user sees in a column at query time.

```sql
CREATE MASKING POLICY mask_email AS
  (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('HR_ADMIN', 'ACCOUNTADMIN') THEN val
    ELSE '****@****.com'
  END;

-- Apply to column
ALTER TABLE employees MODIFY COLUMN email SET MASKING POLICY mask_email;
```

**Key Facts:**

| Aspect | Detail |
|--------|--------|
| Evaluation | Dynamic at query runtime (data is stored unmasked) |
| Input/Output types | Must match — STRING→STRING, NUMBER→NUMBER |
| Per column | Only **ONE** masking policy per column |
| Multi-column | Same policy can be applied to many columns across tables |
| Effect | Transforms/hides the value (not the row) |
| Conditional masking | Can reference other columns using `USING` clause |

```sql
-- Conditional masking: mask based on another column's value
ALTER TABLE t MODIFY COLUMN salary
  SET MASKING POLICY salary_mask USING (salary, department);
```

---

#### Row Access Policy vs Masking Policy — Comparison

| Feature | Row Access Policy | Masking Policy |
|---------|-------------------|----------------|
| Controls | Which **rows** are visible | What **value** is shown in a column |
| Returns | BOOLEAN (show/hide row) | Same data type (masked value) |
| Applied to | Table or View | Column |
| Limit per object | One per table/view | One per column (many columns allowed) |
| Hidden data | Rows completely invisible | Column shows masked/partial value |
| User experience | Fewer rows returned | All rows returned, values redacted |

---

**Q. Can ACCOUNTADMIN bypass row access policies?**
> **Ans:** No. Unlike other privileges, ACCOUNTADMIN does NOT automatically bypass row access policies. The policy itself must explicitly include `CURRENT_ROLE() = 'ACCOUNTADMIN'` as a condition to grant access.

**Q. What's the difference between row-level and column-level security?**
> **Ans:** Row-level security (Row Access Policies) hides entire rows from users. Column-level security (Masking Policies) shows all rows but masks/redacts specific column values based on the user's role.

## Snowflake Last Minute Points

---

### When is a Warehouse Required?

In Snowflake, a warehouse is required only when a statement needs compute resources (read data, write data, run queries, transforms, etc.). Many CREATE commands are metadata-only operations and do not require an active warehouse.

| No Warehouse Needed (Metadata Only) | Warehouse Required (Data Processing) |
|-------------------------------------|--------------------------------------|
| CREATE USER | CREATE TABLE AS SELECT (CTAS) |
| CREATE ROLE | CREATE MATERIALIZED VIEW |
| CREATE DATABASE | INSERT / UPDATE / DELETE |
| CREATE SCHEMA | COPY INTO (load/unload) |
| CREATE STAGE | SELECT queries |
| CREATE FILE FORMAT | MERGE |
| CREATE NETWORK POLICY | Tasks (when executing SQL) |
| GRANT / REVOKE | |

> **Rule:** No data processing = No warehouse. Data processing involved = Warehouse required.

---

### What Can Be Shared via Secure Data Sharing?

| Can Be Shared | Cannot Be Shared |
|---------------|------------------|
| Tables | Regular (non-secure) Views |
| Secure Views | Streams |
| Secure Materialized Views | Tasks |
| Secure UDFs | Stages |
| External Tables | Warehouses |
| | Roles / Users |
| | Pipes |

---

### INFORMATION_SCHEMA

A system-created, read-only schema that Snowflake automatically creates in **every database**. You cannot create user-defined objects inside it. It contains only Snowflake-managed metadata views and table functions.

**Common Views:**
| View | Description |
|------|-------------|
| `INFORMATION_SCHEMA.TABLES` | All tables in the database |
| `INFORMATION_SCHEMA.COLUMNS` | All columns across tables |
| `INFORMATION_SCHEMA.VIEWS` | All views in the database |
| `INFORMATION_SCHEMA.SCHEMATA` | All schemas in the database |
| `INFORMATION_SCHEMA.DATABASES` | All databases (account-wide) |

**Common Table Functions:**
| Function | Description |
|----------|-------------|
| `QUERY_HISTORY()` | Query execution history (7-day retention) |
| `QUERY_HISTORY_BY_USER()` | Query history filtered by user |
| `QUERY_HISTORY_BY_SESSION()` | Query history filtered by session |
| `COPY_HISTORY()` | COPY INTO load history |
| `LOAD_HISTORY()` | Snowpipe load history |
| `TASK_HISTORY()` | Task execution history |
| `PIPE_USAGE_HISTORY()` | Pipe credit usage |

> **Note:** Views are queryable directly (`SELECT * FROM INFORMATION_SCHEMA.TABLES`). Table functions require `TABLE()` wrapper (`SELECT * FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())`).

---

### Unstructured Data URLs in Snowflake

Snowflake provides **three types of URLs** to access files (unstructured data) stored in stages:

| URL Type | Function | Expiry | Access Scope | Use Case |
|----------|----------|--------|-------------|----------|
| **Scoped File URL** | `BUILD_SCOPED_FILE_URL()` | 24 hours | Tied to the user who generated it | Secure, short-lived access for the requesting user |
| **Stage File URL** | `BUILD_STAGE_FILE_URL()` | No expiry | Anyone with stage privileges | Permanent link, requires Snowflake auth |
| **Pre-signed URL** | `GET_PRESIGNED_URL()` | Configurable (default varies) | Anyone with the URL (no auth needed) | Sharing with external users/apps |

**BUILD_SCOPED_FILE_URL:**
```sql
-- Returns a URL scoped to the user; expires in 24 hours
SELECT BUILD_SCOPED_FILE_URL(@my_stage, 'path/to/file.pdf');
```
- URL works only for the user who generated it
- Expires after 24 hours
- Cannot be shared with others
- Requires active Snowflake session to use

**BUILD_STAGE_FILE_URL:**
```sql
-- Returns a permanent stage URL; requires Snowflake authentication
SELECT BUILD_STAGE_FILE_URL(@my_stage, 'path/to/file.pdf');
```
- Permanent (does not expire)
- Requires the user to be authenticated in Snowflake
- Requires privileges on the stage
- Best for internal Snowflake use (e.g., in Snowsight, apps)

**GET_PRESIGNED_URL:**
```sql
-- Returns a pre-signed URL accessible without Snowflake auth; expires after specified seconds
SELECT GET_PRESIGNED_URL(@my_stage, 'path/to/file.pdf', 3600);
-- 3600 = expires in 1 hour
```
- No Snowflake authentication needed to access
- Anyone with the URL can download the file
- Expiry is configurable (seconds)
- Best for sharing files with external users or applications
- **Security risk** if URL is leaked — anyone can access the file until expiry

---

### Exam Q&A

**Q. What is the extension to Snowflake SQL that adds support for procedural logic?**

> **Ans: Snowflake Scripting**
>
> Snowflake Scripting = Language/Extension of Snowflake SQL (DECLARE, BEGIN...END, IF, LOOP, etc.)
>
> Stored Procedure = Object written using Snowflake Scripting (or JavaScript, Python, Java, Scala)

---

**Q. How can you identify queries that might benefit from QAS? (Choose 2)**

- A. By using the `SYSTEM$ESTIMATE_QUERY_ACCELERATION` function
- B. By querying the QUERY_ACCELERATION_CHECK View
- C. By querying the `QUERY_ACCELERATION_ELIGIBLE` View
- D. By checking if there are filters or aggregation in the query

> **Ans: A, C**

---

**Q. What is the effect of adding 0.0.0.0/0 to the blocked IP address list?**

> **Ans:** It blocks all IPv4 addresses — no one can connect (local machine or internet).

---

**Q. What is the default and maximum time retention period for historical data in Snowflake for new accounts??**

> **Ans:** Default retention time: 1 Day. Maximium retention time: 90 Days

---

**Q. How to advance the offset of a stream without consuming change data in DML?**

> **Ans:** Two ways:
> 1. Recreate the stream (`CREATE OR REPLACE STREAM ...`)
> 2. Insert current change data into a temp table with a false condition:
>    ```sql
>    INSERT INTO temp_table SELECT * FROM my_stream WHERE 1=0;
>    ```
>    This advances the offset without inserting any rows.

---

**Q. Difference between `LAST_QUERY_ID()` and `LAST_QUERY_ID(-1)`?**

> **Ans:** No difference. Both return the ID of the most recently executed query in the current session.

**Q. Return the ID of the FIRST query executed in the current session?**

> **Ans:** `LAST_QUERY_ID(1)` — positive numbers count from the start of the session.

| Syntax | Returns |
|--------|---------|
| `LAST_QUERY_ID()` | Most recent query (same as -1) |
| `LAST_QUERY_ID(-1)` | Most recent query |
| `LAST_QUERY_ID(-2)` | Second most recent query |
| `LAST_QUERY_ID(1)` | First query in session |
| `LAST_QUERY_ID(2)` | Second query in session |

---

**Q. What is the purpose of QAS?**

- A. To accelerate parts of the query workload in a warehouse
- B. To reduce the impact of outlier queries
- C. To improve overall warehouse performance
- D. To improve caching

> **Ans: A, B, C** (QAS has nothing to do with caching)

---

**Q. Maximum file size for upload via Snowsight or Classic Console?**

> **Ans: 250 MB**

---

**Q. Types of URLs available for unstructured data in Snowflake?**

- A. Pre-signed URL
- B. Scoped File URL
- C. Stage File URL
- D. HTTPS URL

> **Ans: A, B, C** (there is no generic "HTTPS URL" type in Snowflake)

---

**Q. How can you access a nested element within a JSON object stored in a Snowflake VARIANT column?**

> **Ans: Use a colon (:) followed by the field name

---

**Q. How can you access an element within an array in a Snowflake VARIANT column?**

> **Ans: Use Square brackets ([])

---

**Q. How can you access nested fields or keys within the hierarchical structure in a Snowflake VARIANT column?**

> **Ans: Use dot notation (.) after the initial path.

> Note: Nested access is typically a combination of : and .. The first level commonly starts with : and deeper levels use ..

---

**Q. How can you caste or convert data types in a Snowflake VARIANT column?**

> **Ans: Double colons (::)

```python
# Example for accessing elements in Variant column
SELECT
    data:customer.name::STRING AS customer_name,
    data:customer.address.city::STRING AS city,
    data:orders[0].amount::NUMBER AS first_order_amount
FROM my_table;

```

## Algorithms & Sampling in Snowflake

---

### Approximate & ML Algorithms

| Algorithm | Purpose | Snowflake Function |
|-----------|---------|-------------------|
| **T-Digest** | Approximate percentiles (median, P95, P99) | `APPROX_PERCENTILE()` |
| **HyperLogLog** | Approximate distinct count | `APPROX_COUNT_DISTINCT()` |
| **Space-Saving** | Approximate frequent values / Top-N | `APPROX_TOP_K()` |
| **k-Means** | Clustering / group similar data points | Snowpark ML |
| **XGBoost** | Predict outcomes (classification/regression) | Snowpark ML |
| **Z-Score** | Outlier detection (how far from the mean) | Manual calculation |
| **MinHash** | Compare set similarity without computing intersection/union | `MINHASH()`, `APPROXIMATE_SIMILARITY()` |

**Why use approximate functions?**
- Much faster than exact equivalents on large datasets
- Use significantly less memory
- Acceptable trade-off: small error in exchange for massive speed gain

---

### Sampling Methods

**Purpose:** Return a random subset of rows from a table for exploration, testing, or analysis.

**Syntax:**
```sql
SELECT * FROM my_table SAMPLE (10);        -- 10% of rows (default: Bernoulli)
SELECT * FROM my_table TABLESAMPLE (10);   -- Same as SAMPLE
SELECT * FROM my_table SAMPLE (100 ROWS);  -- Fixed row count
```

**Two Sampling Methods:**

| Method | Alias | How It Works | Speed | Randomness |
|--------|-------|-------------|-------|------------|
| **Bernoulli** | `ROW` | Each row independently has n% chance of inclusion | Slower | More uniform/random |
| **System** | `BLOCK` | Each micro-partition has n% chance of inclusion (all rows in selected partition) | Faster | Less random (block-level) |

**Syntax examples:**
```sql
-- Bernoulli (row-level sampling)
SELECT * FROM my_table SAMPLE BERNOULLI (10);   -- 10% row-by-row
SELECT * FROM my_table SAMPLE ROW (10);          -- same

-- System (block-level sampling)
SELECT * FROM my_table SAMPLE SYSTEM (10);      -- 10% of micro-partitions
SELECT * FROM my_table SAMPLE BLOCK (10);        -- same
```

**Default method (when not specified):** Bernoulli / Row

---

### Seed Value

**What it does:** Makes sampling **deterministic** (reproducible).

```sql
-- With seed: same result every time (on same data)
SELECT * FROM my_table SAMPLE BERNOULLI (10) SEED (42);
SELECT * FROM my_table SAMPLE BERNOULLI (10) REPEATABLE (42);  -- same as SEED
```

| With Seed | Without Seed |
|-----------|-------------|
| Same seed + same data = same sample every time | Different sample each execution |
| Deterministic / reproducible | Random |
| No performance benefit | No performance benefit |

**Key rules:**
- SEED only works with **Bernoulli/Row** method
- SEED does NOT work with System/Block method
- SEED does NOT make sampling faster — it only controls reproducibility
- `SEED` and `REPEATABLE` are interchangeable keywords

---

### Exam Q&A

**Q. What algorithm does Snowflake use to estimate approximate percentile values?**
> **Ans: T-Digest**

**Q. What is the average relative error of Snowflake's HyperLogLog implementation?**
> **Ans: 1.62%**

**Q. What is the purpose of specifying a seed value for sampling?**
> **Ans: To make the sampling deterministic** (same seed + same data = same sample every time)

**Q. Which sampling method is faster on large tables?**
> **Ans: System/Block** (decides at partition level, not row level)

**Q. Which sampling method provides more uniform randomness?**
> **Ans: Bernoulli/Row** (each row evaluated independently)

---

### Quick Summary

| Concept | Remember |
|---------|----------|
| Sampling method | Determines how rows are selected (row vs block) |
| Seed value | Controls reproducibility (not speed) |
| No seed | Always random, different each time |
| Bernoulli | Slower, more random, supports SEED |
| System | Faster, less random, does NOT support SEED |
| Default method | Bernoulli/Row |
| SAMPLE = TABLESAMPLE | Interchangeable keywords |